# MAPNAI — Agent 4 risk labelling (Kaggle)

Labels the sports and entertainment candidate articles with an **event type** and a **severity (0–25)**,
which become the training labels for Agent 4's own DistilRoBERTa risk model. No LLM is used by the
pipeline afterwards — this notebook runs once, offline.

| Step | What it does | Time on 2× T4 |
|---|---|---|
| 1. Install | bitsandbytes + accelerate | 1–2 min |
| 2–5. Setup | Paths, risk taxonomy, prompt, data | < 1 min |
| 6. Load labeller | Qwen2.5-7B-Instruct in 4-bit | 3–5 min (download + load) |
| 7. Label | ~5,300 articles, saved as it goes | 1.5–2 h |
| 8. Retry | Re-asks once for replies that failed validation | 1–3 min |
| 9. Summary | Class balance, severity per type, warnings | < 1 min |
| 10. Gold set | Stratified ~100 articles for **you** to label by hand | < 1 min |
| 11. Package | Zips everything for download | < 1 min |

**Before you start**
1. On Kaggle: **Create → New Dataset** and upload two files from the MAPNAI repo into it:
   `data/classifier/candidates.jsonl` and `data/risk/candidates_extra.jsonl`. Then **Add Input** to attach that
   dataset to this notebook. Both files are found automatically.
2. **Settings → Accelerator → GPU T4 x2**, and **Settings → Internet → On** (needs a phone-verified Kaggle account).
3. **Run All.** It uses Qwen2.5-7B in 4-bit, the same set-up as Agent 2's Colab notebook.

**Resuming:** files in `/kaggle/working` are lost when the session ends unless you **Save Version**. To resume
a stopped run, download `risk_labeled.jsonl` and `risk_rejected.jsonl`, upload them as a dataset and attach it —
already-labelled articles are skipped.

## 1. Install

In [ ]:
# 'hf'   = Qwen2.5-7B-Instruct in 4-bit via transformers (proven on Colab and on Kaggle's Python 3.13 image)
# 'vllm' = Qwen2.5-14B-Instruct-AWQ on both T4s — faster, better labels, but the pinned vLLM needs torch 2.4,
#          which has no build for Kaggle's current Python 3.13 image, so the install fails there
ENGINE = 'hf'

if ENGINE == 'vllm':
    !pip install -q vllm==0.6.3.post1
else:
    !pip install -q -U bitsandbytes accelerate
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Paths and settings

In [ ]:
import glob, json, os, random, re, shutil, time

INPUT, WORK = '/kaggle/input', '/kaggle/working'


def find(name):
    hits = sorted(glob.glob(f'{INPUT}/**/{name}', recursive=True))
    return hits[0] if hits else None


CAND_PATHS = [p for p in (find('candidates.jsonl'), find('candidates_extra.jsonl')) if p]
assert CAND_PATHS, 'candidates.jsonl not found — upload it as a Kaggle dataset and attach it (Add Input).'
LABELED_PATH = f'{WORK}/risk_labeled.jsonl'
REJECTED_PATH = f'{WORK}/risk_rejected.jsonl'

# Resume: copy a previous run's outputs in if they were attached as a dataset
for name, cur in (('risk_labeled.jsonl', LABELED_PATH), ('risk_rejected.jsonl', REJECTED_PATH)):
    prev = find(name)
    if prev and not os.path.exists(cur):
        shutil.copy(prev, cur)
        print('Resuming from', prev)

IN_SCOPE_HINTS = {'sports', 'entertainment_movies'}  # Agent 4 only ever sees these domains
MAX_BODY_CHARS = 1500
LIMIT = None          # e.g. 200 for a quick test run; None = everything
SEED = 42
KEEP_OLD_LABELS = False  # False: labels made with an older prompt version are moved aside and re-labelled
for p in CAND_PATHS:
    print('Candidates:', p)

## 3. Risk taxonomy

Each event type has a **typical severity** (Agent 4's `event_severity` factor, 0–25). The labeller may move it
by up to ±6 for scale (a superstar vs an amateur, career-ending vs minor). Edit here if you want different
values — later, these move into `config/risk_taxonomy.py` so training and Agent 4 share them.

In [ ]:
PROMPT_VERSION = 2   # v2: doping + match_fixing merged into integrity; rules 7-8 added
SEVERITY_SPREAD = 6

# event_type: (typical severity 0-25, definition)
EVENT_TYPES = {
    'routine':        (2,  'results, previews, transfers and signings, releases, trailers, box office, awards, '
                           'reviews, interviews — nothing has gone wrong'),
    'controversy':    (8,  'public criticism, feuds, offensive remarks, fan backlash, disputed decisions, '
                           'boycotts or scandals with no legal action or official sanction yet'),
    'injury':         (12, 'injury, illness or health scare of a player, actor or crew member, '
                           'including being ruled out or sidelined'),
    'legal':          (16, 'lawsuits, arrests, criminal charges, police investigations, court rulings, '
                           'contract or copyright disputes in court'),
    'ban_suspension': (17, 'bans, suspensions, fines or sanctions by a league, governing body, studio, censor '
                           'board or government (doping, fixing and corruption cases are "integrity")'),
    'cancellation':   (18, 'matches, tournaments, events, shoots or releases cancelled, postponed, '
                           'pulled or relocated'),
    'integrity':      (21, 'doping, failed drug tests, anti-doping investigations, match-fixing, spot-fixing, '
                           'illegal betting, bribery or corruption in sport'),
    'death_tragedy':  (25, 'death, fatal accident, stampede, violence, attack or disaster involving '
                           'people in the story'),
}
for t, (sev, _) in EVENT_TYPES.items():
    print(f'{t:15s} {sev:>2d}')

## 4. Prompt

In [ ]:
def build_system_prompt():
    types = '\n'.join(f'- {t} (typical severity {sev}): {desc}' for t, (sev, desc) in EVENT_TYPES.items())
    return f"""You label news articles for a risk-monitoring system that covers SPORTS and FILM/ENTERTAINMENT news.

Pick the ONE event type that is the main subject of the article:
{types}

Rules:
1. If several apply, pick the most severe one that is central to the story, not one mentioned in passing.
2. Speculation, rumours and anniversaries of past events are "routine" unless the article reports a new development.
3. severity: an integer 0-25. Start from the type's typical severity and adjust by at most {SEVERITY_SPREAD}
   for scale — global star or major tournament → higher; amateur, local or minor → lower.
4. in_scope: false if the article is not really about sports or film/entertainment (e.g. politics, finance, music-only).
5. confidence: 0.0-1.0, how sure you are of the event type.
6. evidence: at most 15 words copied from the article that justify the event type.
7. If anyone died, use death_tragedy, even if others were only injured.
8. Reviews, recaps or discussions of a film or TV episode's themes are "routine", whatever the theme.

Reply with ONLY this JSON object and nothing else:
{{"in_scope": true, "event_type": "<one of: {', '.join(EVENT_TYPES)}>", "severity": 0, "confidence": 0.0, "evidence": "..."}}"""


SYSTEM = build_system_prompt()
print(SYSTEM)

## 5. Load the candidate articles

In [ ]:
def load_jsonl(path):
    if not path or not os.path.exists(path):
        return []
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]


# Labels from an older prompt version (and their gold set) are moved aside so everything is labelled the same way
stale = [r for r in load_jsonl(LABELED_PATH) if r.get('prompt_version') != PROMPT_VERSION]
if stale and not KEEP_OLD_LABELS:
    keep = [r for r in load_jsonl(LABELED_PATH) if r.get('prompt_version') == PROMPT_VERSION]
    backup = f'{WORK}/old_labels_v{stale[0].get("prompt_version", 1)}'
    os.makedirs(backup, exist_ok=True)
    for p in [LABELED_PATH, REJECTED_PATH] + glob.glob(f'{WORK}/gold_*'):
        if os.path.exists(p):
            shutil.move(p, os.path.join(backup, os.path.basename(p)))
    with open(LABELED_PATH, 'w', encoding='utf-8') as f:
        f.writelines(json.dumps(r, ensure_ascii=False) + '\n' for r in keep)
    print(f'{len(stale)} labels from an older prompt moved to {backup} — they will be re-labelled.')

cands, seen = [], set()
for p in CAND_PATHS:
    for r in load_jsonl(p):
        if r['id'] not in seen and r.get('hint_domain') in IN_SCOPE_HINTS and (r.get('title') or r.get('body')):
            seen.add(r['id'])
            cands.append(r)
done = {r['id'] for r in load_jsonl(LABELED_PATH)} | {r['id'] for r in load_jsonl(REJECTED_PATH)}
todo = [r for r in cands if r['id'] not in done]
random.Random(SEED).shuffle(todo)   # a partial run is still a representative sample
if LIMIT:
    todo = todo[:LIMIT]
print(f'{len(cands)} in-scope candidates | {len(done)} already done | {len(todo)} to label')

## 6. Load the labeller

`vllm`: Qwen2.5-14B-Instruct-AWQ (~10 GB) split across both T4s. `hf`: Qwen2.5-7B-Instruct in 4-bit (~5.5 GB).

In [ ]:
import torch

if ENGINE == 'vllm':
    from vllm import LLM, SamplingParams
    LABELER = 'Qwen/Qwen2.5-14B-Instruct-AWQ'
    llm = LLM(LABELER, quantization='awq', dtype='float16',
              tensor_parallel_size=torch.cuda.device_count(),
              max_model_len=4096, gpu_memory_utilization=0.90,
              enforce_eager=True,                 # skips CUDA-graph capture: less memory, faster start on T4
              disable_custom_all_reduce=True,     # T4 pairs on Kaggle have no peer-to-peer link
              seed=SEED)
    tok = llm.get_tokenizer()

    def generate(prompts, temperature=0.0):
        params = SamplingParams(temperature=temperature, max_tokens=160, seed=SEED)
        return [o.outputs[0].text for o in llm.generate(prompts, params, use_tqdm=False)]
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    LABELER = 'Qwen/Qwen2.5-7B-Instruct'
    HF_BATCH = 16
    tok = AutoTokenizer.from_pretrained(LABELER, padding_side='left')
    llm = AutoModelForCausalLM.from_pretrained(
        LABELER, device_map='auto',
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                               bnb_4bit_compute_dtype=torch.float16),
    ).eval()

    @torch.no_grad()
    def generate(prompts, temperature=0.0):
        outs = []
        for s in range(0, len(prompts), HF_BATCH):
            enc = tok(prompts[s:s + HF_BATCH], return_tensors='pt', padding=True).to(llm.device)
            gen = llm.generate(**enc, max_new_tokens=160, pad_token_id=tok.eos_token_id,
                               do_sample=temperature > 0, temperature=temperature or None)
            outs += tok.batch_decode(gen[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
        return outs


def to_prompt(r, reminder=''):
    article = json.dumps({'title': r.get('title', ''), 'body': (r.get('body') or '')[:MAX_BODY_CHARS]},
                         ensure_ascii=False)
    return tok.apply_chat_template(
        [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': article + reminder}],
        tokenize=False, add_generation_prompt=True)


print('Labeller ready:', LABELER)

## 7. Label

Every reply is validated: unknown event types or broken JSON go to `risk_rejected.jsonl`; severity is clamped to
the type's typical value ±6. Results are written after every chunk, so stopping and re-running is safe.

In [ ]:
def parse(text):
    m = re.search(r'\{.*\}', text or '', re.S)
    try:
        return json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        return None


def as_bool(v):
    return v.strip().lower() == 'true' if isinstance(v, str) else bool(v)


def validate(d):
    if not isinstance(d, dict):
        return None
    event_type = str(d.get('event_type', '')).strip().lower()
    if event_type not in EVENT_TYPES:
        return None
    base = EVENT_TYPES[event_type][0]
    try:
        severity = int(round(float(d.get('severity', base))))
        confidence = float(d.get('confidence', 0.0))
    except (TypeError, ValueError):
        return None
    return {
        'in_scope': as_bool(d.get('in_scope', True)),
        'event_type': event_type,
        'severity': max(0, base - SEVERITY_SPREAD, min(25, base + SEVERITY_SPREAD, severity)),
        'label_confidence': round(max(0.0, min(1.0, confidence)), 3),
        'evidence': str(d.get('evidence', ''))[:200],
    }


def label(rows, reminder='', temperature=0.0):
    texts = generate([to_prompt(r, reminder) for r in rows], temperature)
    good, bad = [], []
    for r, t in zip(rows, texts):
        lab = validate(parse(t))
        if lab:
            good.append({**r, **lab, 'labeler': LABELER, 'prompt_version': PROMPT_VERSION})
        else:
            bad.append({**r, 'raw_reply': (t or '')[:500]})
    return good, bad


CHUNK = 256 if ENGINE == 'vllm' else 64
t0, n_ok, n_bad = time.time(), 0, 0
with open(LABELED_PATH, 'a', encoding='utf-8') as out, open(REJECTED_PATH, 'a', encoding='utf-8') as rej:
    for s in range(0, len(todo), CHUNK):
        good, bad = label(todo[s:s + CHUNK])
        out.writelines(json.dumps(r, ensure_ascii=False) + '\n' for r in good)
        rej.writelines(json.dumps(r, ensure_ascii=False) + '\n' for r in bad)
        out.flush(); rej.flush()
        n_ok, n_bad = n_ok + len(good), n_bad + len(bad)
        done_n = s + len(todo[s:s + CHUNK])
        rate = done_n / (time.time() - t0)
        print(f'{done_n}/{len(todo)}  ok {n_ok}  rejected {n_bad}  '
              f'{rate:.1f} art/s  ETA {(len(todo) - done_n) / rate / 60:.0f} min')
print(f'Done in {(time.time() - t0) / 60:.1f} min')

## 8. Retry rejected replies once

In [ ]:
rejected = load_jsonl(REJECTED_PATH)
retry = [{k: v for k, v in r.items() if k != 'raw_reply'} for r in rejected if not r.get('retried')]
if retry:
    good, bad = label(retry, reminder='\n\nReply with ONLY the JSON object.', temperature=0.3)
    with open(LABELED_PATH, 'a', encoding='utf-8') as out:
        out.writelines(json.dumps(r, ensure_ascii=False) + '\n' for r in good)
    still_bad = [r for r in rejected if r.get('retried')] + [{**r, 'retried': True} for r in bad]
    with open(REJECTED_PATH, 'w', encoding='utf-8') as f:
        f.writelines(json.dumps(r, ensure_ascii=False) + '\n' for r in still_bad)
    print(f'Retried {len(retry)}: {len(good)} recovered, {len(bad)} still rejected')
else:
    print('Nothing to retry')

## 9. Summary — check before training

Aim for **at least 150 in-scope examples per event type**. If a type is short, collect more articles for it
(e.g. targeted RSS/keyword searches) and run this notebook again — already-labelled articles are skipped.

In [ ]:
import pandas as pd

df = pd.DataFrame(load_jsonl(LABELED_PATH))
scope = df[df.in_scope]
print(f'{len(df)} labelled | {len(scope)} in scope | {(~df.in_scope).sum()} out of scope '
      f'| {len(load_jsonl(REJECTED_PATH))} rejected\n')

summary = (scope.groupby('event_type')
           .agg(n=('id', 'size'), mean_severity=('severity', 'mean'), mean_confidence=('label_confidence', 'mean'))
           .reindex(EVENT_TYPES).fillna(0).round(2))
summary['typical'] = [EVENT_TYPES[t][0] for t in summary.index]
print(summary, '\n')

short = summary[summary.n < 150].index.tolist()
if short:
    print('⚠ Fewer than 150 examples:', ', '.join(short))
print('Low-confidence share (< 0.6):', round((scope.label_confidence < 0.6).mean(), 3))
print('By source domain:\n', pd.crosstab(scope.hint_domain, scope.event_type))

In [ ]:
# Eyeball 3 examples per type — if these look wrong, fix the definitions in step 3 and re-label
pd.set_option('display.max_colwidth', 120)
(scope.sample(frac=1, random_state=SEED).groupby('event_type').head(3).sort_values('event_type')
      [['event_type', 'severity', 'label_confidence', 'title', 'evidence']])

## 10. Gold set for hand-labelling

Picks ~100 in-scope articles, stratified by the labeller's event type (more for the rare, high-stakes types).
`gold_to_label.csv` deliberately **hides the labeller's answer** so it doesn't bias you — fill in
`gold_event_type` (one of the types in step 3) and `gold_in_scope` (TRUE/FALSE) yourself.
The labeller's answers are kept in `gold_key.csv` for scoring later.

These articles are listed in `gold_ids.txt` and **must be excluded from training**.

In [ ]:
GOLD_N = {'routine': 8, 'controversy': 8, 'injury': 11, 'legal': 11, 'ban_suspension': 12,
          'cancellation': 12, 'integrity': 20, 'death_tragedy': 15}

GOLD_CSV, GOLD_KEY, GOLD_IDS = f'{WORK}/gold_to_label.csv', f'{WORK}/gold_key.csv', f'{WORK}/gold_ids.txt'
if os.path.exists(GOLD_IDS):
    print('Gold set already exported — delete gold_ids.txt to draw a new one.')
else:
    gold = pd.concat([scope[scope.event_type == t].sample(min(n, (scope.event_type == t).sum()), random_state=SEED)
                      for t, n in GOLD_N.items()]).sample(frac=1, random_state=SEED)  # shuffle: no type order hints
    gold.assign(gold_event_type='', gold_in_scope='', notes='')[
        ['id', 'title', 'body', 'gold_event_type', 'gold_in_scope', 'notes']].to_csv(GOLD_CSV, index=False)
    gold[['id', 'event_type', 'severity', 'label_confidence']].to_csv(GOLD_KEY, index=False)
    with open(GOLD_IDS, 'w') as f:
        f.write('\n'.join(gold.id))
    print(f'{len(gold)} gold articles →', GOLD_CSV)
    print(gold.event_type.value_counts().to_string())

## 11. Package the outputs

In [ ]:
import zipfile

files = [p for p in (LABELED_PATH, REJECTED_PATH, GOLD_CSV, GOLD_KEY, GOLD_IDS) if os.path.exists(p)]
zip_path = f'{WORK}/mapnai_risk_labels.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in files:
        z.write(p, os.path.basename(p))
print('Saved', zip_path)
for p in files:
    print(f'  {os.path.basename(p):22s} {os.path.getsize(p) / 1024:,.0f} KB')

**Download** `mapnai_risk_labels.zip` from the **Output** panel on the right (or **Save Version** to keep it on Kaggle).


**On your PC**, unzip it into `data/risk/` in the MAPNAI repo. Then:
1. Fill in `gold_to_label.csv` by hand (~1–1.5 h) and save it as `gold_labeled.csv`.
2. Train the DistilRoBERTa risk model on `risk_labeled.jsonl` — in-scope rows only, excluding `gold_ids.txt`.